# VTuber Commercial Pipeline

이 노트북은 Google Colab에서 VTuber 아바타와 악세사리를 제작하기 위한 파이프라인입니다.

**두 가지 모드를 지원합니다:**

1. **Avatar Mode**: 캐릭터 이미지 → 3D VRM 아바타
2. **Accessory Mode**: 악세사리 이미지 → GLB 악세사리 + 장착 메타데이터

> 이미지 생성 AI는 이 노트북에 포함하지 않습니다. 입력 이미지는 사용자가 외부에서 생성해 가져옵니다.


## Setup - 환경 설정


In [ ]:
# TripoSR 설치 (Colab)
import subprocess, sys
subprocess.run(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'TripoSR', '-q'], check=False)


In [ ]:
# Virtual-pipeline 패키지 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/jujumelona/Virtual-pipeline.git', '-q'], check=False)


In [ ]:
# 패키지 임포트 테스트
from vtuber_pipeline.core.config import BLOCKED_PACKAGES
from vtuber_pipeline.core.cache import PipelineCache
from vtuber_pipeline.avatar.face_detector import AnimeFaceDetector
from vtuber_pipeline.avatar.reconstruction import reconstruct_avatar
from vtuber_pipeline.accessory.attachment import AnchorType, generate_attachment_config
print('패키지 임포트 성공!')
print('차단된 패키지:', list(BLOCKED_PACKAGES.keys()))


---

# Avatar Mode - 아바타 모드

외부 이미지 생성 AI에서 만든 캐릭터 이미지를 VRM 아바타로 변환합니다.


## 입력 이미지 업로드


In [ ]:
# 캐릭터 이미지 업로드
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, '캐릭터 이미지를 업로드하세요.'
name = next(iter(uploaded))
Path('/content/input').mkdir(exist_ok=True)
avatar_image = Path('/content/input') / name
avatar_image.write_bytes(uploaded[name])
print('업로드된 아바타 이미지:', avatar_image)


## 아바타 파이프라인 실행


In [ ]:
# 아바타 3D 재구성 실행
from vtuber_pipeline.avatar.reconstruction import reconstruct_avatar
from vtuber_pipeline.avatar.face_detector import AnimeFaceDetector

# 얼굴 검출 및 랜드마크 추출
detector = AnimeFaceDetector()
landmarks = detector.detect(str(avatar_image))
print(f'검출된 얼굴 수: {len(landmarks)}')

# 3D 재구성
result = reconstruct_avatar(
    image_path=str(avatar_image),
    output_dir='/content/workspace/avatar',
    profile='commercial'
)
print('재구성 완료:', result)

# TODO: 템플릿 피팅 (template fitting)
# TODO: VRM 내보내기 (VRM export)


---

# Accessory Mode - 악세사리 모드

완성된 VRM에 장착할 악세사리를 생성합니다.


## 악세사리 이미지 업로드


In [ ]:
# 악세사리 이미지 업로드 (여러 개 가능)
from google.colab import files
from pathlib import Path

uploaded_acc = files.upload()
acc_dir = Path('/content/input/accessories')
acc_dir.mkdir(parents=True, exist_ok=True)
accessory_images = []
for name, data in uploaded_acc.items():
    p = acc_dir / name
    p.write_bytes(data)
    accessory_images.append(p)
print('업로드된 악세사리:', accessory_images)


## 악세사리 배치 재구성


In [ ]:
# 악세사리 일괄 3D 재구성
from vtuber_pipeline.accessory.batch import batch_reconstruct_accessories

if accessory_images:
    results = batch_reconstruct_accessories(
        image_paths=[str(p) for p in accessory_images],
        output_dir='/content/workspace/accessories',
        profile='commercial'
    )
    print('재구성된 악세사리:', results)


## 장착 설정 생성


In [ ]:
# 장착 설정 생성 예시
from vtuber_pipeline.accessory.attachment import AnchorType, generate_attachment_config

# 지원 앵커 타입:
# HEAD_TOP: 왕관, 모자, 고양이귀
# FACE: 안경, 마스크
# LEFT_EAR, RIGHT_EAR
# NECK
# CHEST
# BACK: 날개, 가방
# LEFT_HAND, RIGHT_HAND
# HIPS: 꼬리 root 등

attachment_config = generate_attachment_config(
    name='crown',
    anchor=AnchorType.HEAD_TOP,
    translation=[0.0, 0.13, 0.01],
    rotation=[0.0, 0.0, 0.0],
    scale=[0.11, 0.11, 0.11]
)
print('장착 설정:', attachment_config)


---

# Scaffold Fallback - 기본 구현

패키지의 고급 기능이 아직 구현되지 않은 경우 사용할 수 있는 기본 구현입니다.

상업 프로필은 permissive/commercial-friendly stack만 사용합니다. InstantMesh는 `nvdiffrast` 라이선스 때문에 기본적으로 금지합니다.


## 외부 이미지 생성용 권장 Prompt

### Avatar

```text
[CHARACTER DESCRIPTION]

Create a clean VTuber character reference image for 3D reconstruction.
Single character only. Front-facing or near-front-facing camera.
Head, hair silhouette, neck, shoulders and upper torso fully visible.
Waist-up composition preferred. Neutral expression.
Eyes and mouth clearly visible. Arms not blocking torso.
No large prop covering the body. Simple uniform background.
Even lighting. Minimal perspective distortion. High facial detail.
```

### Accessory

```text
[ACCESSORY DESCRIPTION]

Create one isolated wearable or handheld accessory for 3D reconstruction.
Single object only. Fully visible. Centered. Plain uniform background.
No person, no mannequin, no hand holding it, no occlusion.
Neutral canonical orientation. Even lighting. Minimal perspective distortion.
```


## Runtime 확인


In [ ]:
# 런타임 환경 확인
import os, subprocess, json, platform

print('Python:', platform.python_version())
subprocess.run(['nvidia-smi'], check=False)
print('RAM/Disk:')
subprocess.run(['bash','-lc','free -h && df -h /content'], check=False)


## 상업 프로필 안전장치


In [ ]:
# 상업적 사용이 제한된 패키지 목록
BLOCKED = {
    'instantmesh': 'runtime path uses nvdiffrast',
    'nvdiffrast': 'public license restricts non-NVIDIA use to non-commercial research/evaluation',
    'stable-fast-3d': 'community license has commercial conditions/revenue threshold',
}
print(json.dumps(BLOCKED, ensure_ascii=False, indent=2))


## 필수 도구 설치 — 계획 셀


In [ ]:
# 실제 구현 시 commit/revision을 반드시 pin한다.
!pip -q install -U pip setuptools wheel
!pip -q install anime-face-detector pydantic pyyaml trimesh pillow

# TripoSR: code + pretrained model are MIT according to upstream.
%cd /content
!mkdir -p third_party
%cd /content/third_party
!test -d TripoSR || git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR
!pip -q install -r requirements.txt

# 중요: commercial profile에서 TripoSR는 --no-remove-bg 사용.


## 얼굴/랜드마크 분석 (기본 구현)


In [ ]:
# 기본 얼굴 검출 구현
import cv2
from anime_face_detector import create_detector

img = cv2.imread(str(avatar_image))
assert img is not None

detector = create_detector('yolov3')
preds = detector(img)
print('검출된 얼굴 수:', len(preds))
if len(preds) == 0:
    raise RuntimeError('애니 얼굴을 검출하지 못했습니다. 입력 프롬프트 조건에 맞춰 이미지를 다시 생성하세요.')

print('랜드마크 타입:', type(preds[0]), preds[0].keys() if hasattr(preds[0], 'keys') else '')


## TripoSR 3D reference reconstruction (기본 구현)


In [ ]:
# TripoSR로 3D 재구성
from pathlib import Path
import subprocess

out_dir = Path('/content/workspace/avatar/reconstruct')
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', '/content/third_party/TripoSR/run.py',
    str(avatar_image),
    '--no-remove-bg',
    '--device', 'cuda:0',
    '--chunk-size', '8192',
    '--mc-resolution', '256',
    '--model-save-format', 'glb',
    '--output-dir', str(out_dir),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('참조 재구성 완료:', out_dir)


## Template fitting / facial rig / VRM (계획)

여기부터가 이 프로젝트의 핵심 구현부입니다. 단순히 TripoSR mesh를 VRM으로 내보내지 않습니다.

```text
TripoSR reference
+ anime facial landmarks
+ fixed CC0 VTuber face/body template
        ↓
non-rigid fitting (topology/index 유지)
        ↓
texture projection
        ↓
fixed humanoid skeleton + skin weights
        ↓
blink / aa-ih-ou-ee-oh / emotion / gaze
        ↓
hair SpringBone
        ↓
VRM export
```


In [ ]:
# 계획된 최종 API — 현재 scaffold에서는 구현 전.
# from vtuber_pipeline import build_avatar
# result = build_avatar(
#     image=str(avatar_image),
#     profile='commercial',
#     mode='streaming',
#     output='/content/outputs/avatar',
# )
# print(result)


## Avatar validation — 최종 구현 목표


In [ ]:
# 아바타 검증 체크리스트
required_checks = [
    'VRM parse',
    'required humanoid bones',
    'head/neck rotation',
    'blink / blinkLeft / blinkRight',
    'aa / ih / ou / ee / oh',
    'happy / angry / sad / relaxed / surprised',
    'eye gaze',
    'springbone references',
    'textures/materials',
    'front + 45deg + expression preview renders',
]
for x in required_checks:
    print('[ ]', x)


## Accessory Mode — batch 3D reconstruction (기본 구현)


In [ ]:
# TripoSR로 악세사리 일괄 3D 재구성
# commercial profile: --no-remove-bg를 유지한다.
if accessory_images:
    out = Path('/content/workspace/accessories/reconstruct')
    out.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', '/content/third_party/TripoSR/run.py',
        *[str(p) for p in accessory_images],
        '--no-remove-bg',
        '--device', 'cuda:0',
        '--chunk-size', '8192',
        '--mc-resolution', '256',
        '--model-save-format', 'glb',
        '--output-dir', str(out),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


## Accessory attachment configuration (기본 예시)


In [ ]:
# 장착 설정 예시
example_attachment = {
    'schema': 'vtuber-accessory-attachment/1',
    'name': 'black_crown',
    'anchor': 'HEAD_TOP',
    'parent_bone': 'head',
    'translation': [0.0, 0.13, 0.01],
    'rotation_euler': [0.0, 0.0, 0.0],
    'scale': [0.11, 0.11, 0.11],
    'physics': None,
}
print(json.dumps(example_attachment, indent=2))


## Accessory output policy

기본 출력은 악세사리를 본체에 강제로 합치지 않습니다.

```text
crown.glb
crown/attachment.json
crown/preview.png
```

필요한 경우에만:

```text
character + crown → character_with_crown.vrm
```

을 생성합니다. 여러 악세사리 조합의 VRM을 기본으로 전부 만들면 조합 수가 폭발하므로 optional입니다.


## Resume / Drive cache

최종 구현에서는 각 stage 뒤에 `state.json`, `manifest.json`, SHA256을 Drive에 저장합니다. Colab 연결이 끊겨도 완료된 3D reconstruction부터 다시 시작하지 않습니다.
